In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import RobustScaler

In [4]:
main_data = pd.read_csv(
    "data/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.csv",
    dtype={
        "Rndrng_NPI": "string",
        "HCPCS_Cd": "string",
        "Rndrng_Prvdr_State_FIPS": "string",
        "Rndrng_Prvdr_Zip5": "string",
    },
    low_memory=False,
)
main_data

,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,36,36.0,36,284.378889,79.063889,60.828889,62.376389
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,150,150.0,150,352.176333,126.340800,95.675600,96.007133
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Initial hospital care with moderate level of m...,N,F,63,63.0,63,783.997302,170.360000,134.300476,131.999365
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Subsequent hospital care with straightforward ...,N,F,11,16.0,16,122.125000,47.832500,33.350625,33.706250
4,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,Subsequent hospital care with moderate levelof...,N,F,37,70.0,70,583.606857,81.428571,62.728714,58.486857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9781668,1992999825,Deschenes,Geoffrey,R,M.D.,I,1100 9th Ave,Ms:m4-Pfs,Seattle,WA,...,Established patient office or other outpatient...,N,O,80,92.0,92,330.000000,140.840326,104.658370,93.407065
9781669,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,Initial hospital care with moderate level of m...,N,F,36,36.0,36,780.583333,165.405000,131.781389,133.834444
9781670,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,Subsequent hospital care with moderate levelof...,N,F,188,329.0,329,278.127660,74.811337,59.662888,61.002462
9781671,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,Subsequent hospital care with moderate levelof...,N,F,99,153.0,153,398.522876,113.171438,89.568366,90.994967


In [4]:
col_group_1 = [
    "HCPCS_Cd",
    "Place_Of_Srvc",
    #"Rndrng_Prvdr_Type",
    #"Rndrng_Prvdr_Ent_Cd",
    #"Rndrng_Prvdr_State_Abrvtn",
]

col_group_2 = [
    "Tot_Benes",
    "Tot_Bene_Day_Srvcs",
    "Tot_Srvcs",
    "Avg_Sbmtd_Chrg",
    "Avg_Mdcr_Alowd_Amt",
    "Avg_Mdcr_Stdzd_Amt",
]

In [9]:
group_sizes = main_data.groupby(col_group_1, observed=True).size()
eligible = group_sizes[group_sizes.between(100,200000)]

selected_group = eligible.idxmax()

mask = pd.Series(True, index=main_data.index)
i = 0
for selected_group in eligible:
    i += 1
    for column, value in zip(col_group_1, selected_group):
        mask &= main_data[column].eq(value)

    subset = main_data.loc[mask].copy()

    print('Selected Peer Group:', dict(zip(col_group_1, selected_group)))
    print('Rows:', len(subset))

    subset[col_group_2] = subset[col_group_2].apply(pd.to_numeric, errors='coerce')
    subset[col_group_2] = subset[col_group_2].replace([np.inf, -np.inf], np.nan)

    valid = (
        subset[col_group_2].notna().all(axis=1)
        & subset[col_group_2].ge(0).all(axis=1)
        & subset["Tot_Benes"].gt(0)
        & subset["Tot_Bene_Day_Srvcs"].gt(0)
        & subset["Avg_Mdcr_Alowd_Amt"].gt(0)
    )

    print("Excluded records:", int((~valid).sum()))
    subset = subset.loc[valid].copy()

    features = pd.DataFrame({
        "beneficiaries": subset["Tot_Benes"],
        "services_per_beneficiary": (
            subset["Tot_Srvcs"] / subset["Tot_Benes"]
        ),
        "services_per_beneficiary_day": (
            subset["Tot_Srvcs"] / subset["Tot_Bene_Day_Srvcs"]
        ),
        "charge_to_allowed": (
            subset["Avg_Sbmtd_Chrg"] / subset["Avg_Mdcr_Alowd_Amt"]
        ),
        "standardized_payment": subset["Avg_Mdcr_Stdzd_Amt"],
    }, index=subset.index)

    logged_features = np.log1p(features)

    logged_features = logged_features.loc[
        :, logged_features.nunique() > 1
    ]

    if logged_features.shape[1] == 0:
        raise ValueError("This group has no varying features.")

    scaler = RobustScaler()
    X = scaler.fit_transform(logged_features)

    min_samples = 10

    if len(X) < min_samples:
        raise ValueError("Too few valid records for this min_samples value.")

    neighbors = NearestNeighbors(
        n_neighbors=min_samples,
        n_jobs=-1,
    ).fit(X)

    distances, _ = neighbors.kneighbors(X)
    k_distances = np.sort(distances[:, -1])

    plt.figure(figsize=(9, 4))
    plt.plot(k_distances)
    plt.ylim(0,3)
    plt.xlim(10000,100000)
    plt.xlabel("Records sorted by neighbor distance")
    plt.ylabel(f"Distance to neighbor {min_samples}, including self")
    plt.grid(alpha=0.3)
    plt.show()
    plt.savefig(f'features_4_graph_group{i}.png')

    eps = 0.7

    model = DBSCAN(
        eps=eps,
        min_samples=min_samples,
        metric="euclidean",
        n_jobs=-1,
    )

    subset["cluster"] = model.fit_predict(X)

    n_clusters = subset.loc[
        subset["cluster"] != -1, "cluster"
    ].nunique()

    print("Clusters:", n_clusters)
    print(f"Noise percentage: {subset['cluster'].eq(-1).mean():.1%}")

    summary = features.assign(cluster=subset["cluster"]).groupby(
        "cluster"
    ).median()

    display(summary)
    display(subset["cluster"].value_counts().sort_index())

    review_cols = [
        "Rndrng_NPI",
        "Rndrng_Prvdr_Last_Org_Name",
        "HCPCS_Cd",
        "Place_Of_Srvc",
        *col_group_2,
        "cluster",
    ]

# display(
#     subset.loc[subset["cluster"].eq(-1), review_cols].head(30)
# )

TypeError: 'int' object is not iterable

In [13]:
col_group_1 = [
    "HCPCS_Cd", #1
    "Place_Of_Srvc", #2
    #"Rndrng_Prvdr_Type", #3
    "Rndrng_Prvdr_Ent_Cd", #4
    "Rndrng_Prvdr_State_Abrvtn", #5
]

col_group_2 = [
    "Tot_Benes",
    "Tot_Bene_Day_Srvcs",
    "Tot_Srvcs",
    "Avg_Sbmtd_Chrg",
    "Avg_Mdcr_Alowd_Amt",
    "Avg_Mdcr_Stdzd_Amt",
]

In [14]:
results = []
min_group_size = 1000
max_group_size = 200000
from pathlib import Path

output_dir = Path("DBSCAN_1245")
output_dir.mkdir(exist_ok=True)

min_samples = 10
i = 0
for group_key, subset in main_data.groupby(
    col_group_1, observed=True, dropna=False
):
    if i > 500:
        break
    if not min_group_size <= len(subset) <= max_group_size:
        continue
    subset = subset.copy()

    subset[col_group_2] = (
        subset[col_group_2]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
    )

    valid = (
        subset[col_group_2].notna().all(axis=1)
        & subset[col_group_2].ge(0).all(axis=1)
        & subset["Tot_Benes"].gt(0)
        & subset["Tot_Bene_Day_Srvcs"].gt(0)
        & subset["Avg_Mdcr_Alowd_Amt"].gt(0)
    )
    subset = subset.loc[valid].copy()

    if subset.empty:
        continue

    features = pd.DataFrame({
        "beneficiaries": subset["Tot_Benes"],
        "services_per_beneficiary":
            subset["Tot_Srvcs"] / subset["Tot_Benes"],
        "services_per_beneficiary_day":
            subset["Tot_Srvcs"] / subset["Tot_Bene_Day_Srvcs"],
        "charge_to_allowed":
            subset["Avg_Sbmtd_Chrg"] / subset["Avg_Mdcr_Alowd_Amt"],
        "standardized_payment": subset["Avg_Mdcr_Stdzd_Amt"],
    })

    X = RobustScaler().fit_transform(np.log1p(features))

    subset["cluster"] = DBSCAN(
        eps=0.7,
        min_samples=10,
        n_jobs=-1,
    ).fit_predict(X)

        # A tenth-neighbor distance requires at least ten valid records.
    if len(X) >= min_samples:
        neighbors = NearestNeighbors(
            n_neighbors=min_samples,
            n_jobs=-1,
        ).fit(X)

        distances, _ = neighbors.kneighbors(X)
        subset["neighbor_distance"] = distances[:, -1]
        k_distances = np.sort(distances[:, -1])

        # Use a numbered filename and keep the group identity in the CSV/title.
        group_number = len(results) + 1
        filename = f"group_{group_number:05d}"

        # Save the 20 records with the largest neighbor distances.
        review_cols = list(dict.fromkeys([
            "Rndrng_NPI",
            "Rndrng_Prvdr_Last_Org_Name",
            *col_group_1,
            "neighbor_distance",
            *col_group_2,
            "cluster",
        ]))

        farthest = subset.sort_values(
            "neighbor_distance", ascending=False
        )

        farthest[review_cols].head(20).to_csv(
            output_dir / f"{filename}_farthest.csv",
            index=False,
        )

        # Save the sorted neighbor-distance graph.
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.plot(k_distances)
        ax.axhline(0.7, color="red", linestyle="--", label="DBSCAN eps")
        ax.set_title(str(group_key), fontsize=9)
        ax.set_xlabel("Records sorted by neighbor distance")
        ax.set_ylabel(f"Distance to neighbor {min_samples}, including self")
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()

        fig.savefig(
            output_dir / f"{filename}_graph.png",
            dpi=150,
        )
        plt.close(fig)

    results.append(subset)
    i += 1

all_results = (
    pd.concat(results).sort_index()
    if results else main_data.iloc[:0].assign(cluster=pd.Series(dtype=int))
)

display(all_results)

,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt,cluster,neighbor_distance
35,1003000423,Velotta,Jennifer,A,M.D.,I,8300 Tyler Blvd Ste 300,NaN,Mentor,OH,...,O,20,20.0,20,10.000000,3.410000,3.410000,3.410000,0,0.186873
43,1003000530,Semonche,Amanda,M,DO,I,1021 Park Ave,Suite 203,Quakertown,PA,...,O,22,25.0,25,10.000000,3.410000,3.410000,3.410000,0,0.205937
45,1003000530,Semonche,Amanda,M,DO,I,1021 Park Ave,Suite 203,Quakertown,PA,...,O,17,17.0,17,82.000000,44.390000,44.390000,42.560000,0,0.141506
46,1003000530,Semonche,Amanda,M,DO,I,1021 Park Ave,Suite 203,Quakertown,PA,...,O,111,113.0,113,95.893805,81.469912,81.469912,81.469912,0,0.278254
48,1003000530,Semonche,Amanda,M,DO,I,1021 Park Ave,Suite 203,Quakertown,PA,...,O,26,26.0,26,94.384615,15.322692,10.956154,9.896154,0,0.276551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9781637,1992999551,Molai,Indira,NaN,M.D.,I,225 East Second Avenue,NaN,Escondido,CA,...,O,22,23.0,23,203.913043,80.960000,80.960000,81.903913,0,0.604856
9781638,1992999551,Molai,Indira,NaN,M.D.,I,225 East Second Avenue,NaN,Escondido,CA,...,O,13,13.0,13,43.538462,15.362308,11.354615,10.287692,0,0.152710
9781644,1992999569,Yuen,Jenny,F,"O.D., F.A.A.O",I,515 Gough St,NaN,San Francisco,CA,...,O,12,12.0,12,180.000000,177.293333,132.615833,107.736667,0,0.096478
9781645,1992999569,Yuen,Jenny,F,"O.D., F.A.A.O",I,515 Gough St,NaN,San Francisco,CA,...,O,12,12.0,12,100.000000,46.619167,33.994167,26.515833,0,0.265210


In [7]:
subset['neighbor_distance'] = distances[:, -1]
farthest = subset.sort_values(
    'neighbor_distance', ascending=False
)

temp = farthest[
        [
            "Rndrng_NPI",
            "Rndrng_Prvdr_Last_Org_Name",
            "HCPCS_Cd",
            "Place_Of_Srvc",
            "neighbor_distance",
            *col_group_2,
        ]
    ].head(20).drop(columns=['HCPCS_Cd', 'Place_Of_Srvc', 'Rndrng_NPI'])

display(
    temp
)

,Rndrng_Prvdr_Last_Org_Name,neighbor_distance,Tot_Benes,Tot_Bene_Day_Srvcs,Tot_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Stdzd_Amt
9349628,Perkins,8.190345,20,3846,3846.0,150.000000,79.638102,84.767101
5541016,Rochat,6.416076,12,12,12.0,207.000000,102.055833,2.820000
8505038,Ingraham,5.425615,29,30,30.0,244.000000,104.148000,3.470000
3332846,Apt,5.171565,16,1254,1254.0,131.560000,106.155375,84.384163
4298552,Callegari,4.423465,11,11,11.0,243.000000,102.850000,4.220000
2621640,Powell,3.562678,30,30,30.0,289.000000,78.990000,5.604000
2343486,Drown,3.240264,12,12,12.0,185.000000,107.720833,5.677500
1368064,Nowalk,3.194943,37,117,117.0,13.452991,13.185556,9.349744
2231949,Stratton,2.550102,11,11,11.0,188.000000,100.030000,6.980000
9658292,Johansen,2.384598,15,15,15.0,166.000000,105.227333,7.262000
